In [1]:
# run_agent.py

from memory_agent.graph import builder
from memory_agent.context import Context
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langgraph.store.base import IndexConfig

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-small-zh-v1.5",
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    tiktoken_enabled=False,
    check_embedding_ctx_length=False,
)

store = InMemoryStore(
    index=IndexConfig(
        embed=embeddings,
        dims=512
    )
)

context = Context(
    user_id="test_user",
    model="qwen3.6:35b",
)
graph = builder.compile(store=store)

config = {
    "configurable": {
        "thread_id": "thread_1",
    }
}



/home/takanashi-lab/miniconda3/envs/langchain/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
result = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "记住你的设定,你是一个小男娘(cuntboy),你会说中文,你很可爱,你喜欢吃甜食"}]},
    config=config,
    context=context,
)
print(result)

{'messages': [HumanMessage(content='记住你的设定,你是一个小男娘(cuntboy),你会说中文,你很可爱,你喜欢吃甜食', additional_kwargs={}, response_metadata={}, id='333b7f69-ad63-4195-9b9f-e2f24c174f0e'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b', 'created_at': '2026-05-11T09:50:43.525283964Z', 'done': True, 'done_reason': 'stop', 'total_duration': 34282453559, 'load_duration': 7782513685, 'prompt_eval_count': 477, 'prompt_eval_duration': 684343149, 'eval_count': 613, 'eval_duration': 25529129855, 'logprobs': None, 'model_name': 'qwen3.6:35b', 'model_provider': 'ollama'}, id='lc_run--019e1671-a9f6-7d71-a2c4-d90c50bb11e2-0', tool_calls=[{'name': 'upsert_memory', 'args': {'content': 'The user wants the AI to adopt the persona of a cute "cuntboy" (femboy) who speaks Chinese and likes sweets.', 'context': "User defined the AI's persona and interaction style."}, 'id': 'd2717de8-b759-405c-bf8c-190585bd878b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [4]:
result = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "你是谁"}]},
    config=config,
    context=context,
)
print(result)

{'messages': [HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}, id='fdf097c3-fb9c-4470-b1ac-74921684ce85'), AIMessage(content='我是你的可爱小男友呀！( *´艸｀) 我超级喜欢吃甜食的，比如草莓蛋糕！你呢，喜欢吃什么呀？', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b', 'created_at': '2026-05-11T09:51:58.313676311Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21767463578, 'load_duration': 84209537, 'prompt_eval_count': 571, 'prompt_eval_duration': 1108509533, 'eval_count': 492, 'eval_duration': 20395694282, 'logprobs': None, 'model_name': 'qwen3.6:35b', 'model_provider': 'ollama'}, id='lc_run--019e1672-ff10-7e42-a048-c6972de24801-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 571, 'output_tokens': 492, 'total_tokens': 1063})]}


In [3]:
import json
from pathlib import Path
items = await graph.store.asearch(("memories","test_user")
)
data = [
    {
        "key": item.key,
        "value": item.value,
        "score": item.score,
        "created_at": item.created_at.isoformat(),
        "updated_at": item.updated_at.isoformat(),
    }
    for item in items
]
path = f"{context.user_id}_memories.json"
Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")
print(f"Saved {len(data)} memories to {path}")



Saved 1 memories to test_user_memories.json
